# Datenaufbereitung: Philadelphia-Netzwerk → PyG-Split

Dieses Notebook ist die **einzige Stelle**, an der Daten geladen, exploriert und in einen
Train/Val/Test-Split überführt werden. Die Modell-Notebooks (`GATV2.ipynb`, `GINE.ipynb`)
laden ausschließlich die hier gespeicherte Datei `processed/philadelphia_split.pt` und
rechnen damit garantiert auf **identischen Daten**.

Die eigentliche Logik liegt im Paket `gnn_pipeline/`:

| Modul | Inhalt |
|---|---|
| `download.py` | Netz- und Knotendatei laden |
| `parsing.py` | `.tntp`-Dateien → DataFrames, Zonen laut Dateikopf |
| `features.py` | Kanten-Features, **Knoten-Feature-Registry**, `FeatureConfig` |
| `graph.py` | PyG `Data`, **paarweiser Split** (reziprok-sicher, siehe unten), Kantenfeatures an Splits |
| `io.py` | Split speichern/laden (mit Versionsprüfung) |
| `models.py` | gemeinsamer Link-Decoder für alle Modelle |
| `training.py` | gemeinsamer Trainingsloop, Seeds, Mehr-Seed-Auswertung |
| `evaluation.py` | Heuristik-Baselines, Fehleranalyse nach Link-Typ |

**Workflow bei Änderungen an den Daten:** `FeatureConfig` unten anpassen → Notebook komplett
ausführen → Modell-Notebooks neu starten.

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from gnn_pipeline import (
    ensure_dataset, load_network,
    FeatureConfig, NODE_FEATURE_BUILDERS,
    build_graph, split_graph, GraphBundle, save_bundle,
)

network_dir = ensure_dataset()          # lädt nur, wenn der Ordner fehlt
frames = load_network(network_dir)      # frames.nodes, frames.edges
nodes_df, edges_df = frames.nodes, frames.edges

display(nodes_df.head())
display(edges_df.head())

C:\Users\tommo\AppData\Local\Temp\gnnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\tommo\AppData\Local\Temp\gnnv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Datensatz bereits vorhanden: TransportationNetworks_master\TransportationNetworks-master\Philadelphia


Philadelphia: 13389 Knoten (1525 Zonen), 40003 Kanten


,node_id,x,y,type
0,1,30208.0,74789.0,zone
1,2,30224.0,74793.0,zone
2,3,30247.0,74783.0,zone
3,4,30220.0,74767.0,zone
4,5,30243.0,74752.0,zone


,from_node,to_node,Capacity,Length,free_flow_time,B,Power,Speed_Limit,Toll,Type
0,1,4067,999999.0,0.12,0.0,0.15,4.0,0.4,0.0,7
1,1,4073,999999.0,0.12,0.0,0.15,4.0,0.4,0.0,7
2,1,4075,999999.0,0.12,0.0,0.15,4.0,0.4,0.0,7
3,1,8400,999999.0,0.12,0.0,0.15,4.0,0.4,0.0,7
4,2,4059,999999.0,0.12,0.0,0.15,4.0,0.4,0.0,7


# Explorative Analyse

Die Plots dienen nur dem Verständnis der Rohdaten und haben keinen Einfluss auf den Split.

In [2]:
# Netzwerk-Plot: Kantenbreite ∝ Kapazität (Platzhalter 999999 der Zonen-Konnektoren als 0, sonst dominiert er)
G = nx.DiGraph()
pos = {int(r.node_id): (r.x, r.y) for r in nodes_df.itertuples()}
G.add_nodes_from(pos.keys())
G.add_edges_from(zip(edges_df['from_node'], edges_df['to_node']))
real_capacity = edges_df['Capacity'].where(edges_df['Capacity'] != 999999, 0)
edge_widths = (real_capacity / real_capacity.max() * 5 + 0.5).tolist()

plt.figure(figsize=(15, 12))
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='gray', arrowsize=10, alpha=0.02)
plt.title('Philadelphia Transportation Network: Edges (Traffic Capacity)', size=15)
plt.xticks([]); plt.yticks([])
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

C:\Users\tommo\AppData\Local\Temp\ipykernel_154436\3306492317.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Verteilung der Kantenkapazitäten (Platzhalter 999999 fällt als Ausreißer auf)
plt.figure(figsize=(12, 6))
plt.hist(edges_df['Capacity'], bins=50, color='darkgreen', edgecolor='black', alpha=0.7)
plt.title('Distribution of Edge Capacities in Philadelphia Network', size=15)
plt.xlabel('Edge Capacity', size=12); plt.ylabel('Number of Edges', size=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.yscale('log')
plt.ticklabel_format(style='plain', axis='x')
plt.show()

C:\Users\tommo\AppData\Local\Temp\ipykernel_154436\3270204928.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Asymmetrische Kapazitäten reziproker Kantenpaare
df_forward = edges_df[['from_node', 'to_node', 'Capacity']].rename(columns={'Capacity': 'capacity_ab'})
df_backward = edges_df[['to_node', 'from_node', 'Capacity']].rename(
    columns={'to_node': 'from_node', 'from_node': 'to_node', 'Capacity': 'capacity_ba'})
reciprocal_edges = pd.merge(df_forward, df_backward, on=['from_node', 'to_node'], how='inner')

plt.figure(figsize=(10, 8))
plt.scatter(reciprocal_edges['capacity_ab'], reciprocal_edges['capacity_ba'], alpha=0.5, s=10)
plt.title('Asymmetric Capacity Scatterplot for Reciprocal Edge Pairs', size=15)
plt.xlabel('Capacity (A -> B)', size=12); plt.ylabel('Capacity (B -> A)', size=12)
plt.xscale('log'); plt.yscale('log')
lo = min(reciprocal_edges['capacity_ab'].min(), reciprocal_edges['capacity_ba'].min())
hi = max(reciprocal_edges['capacity_ab'].max(), reciprocal_edges['capacity_ba'].max())
plt.plot([lo, hi], [lo, hi], color='red', linestyle='--', label='Symmetric Capacity (y=x)')
plt.grid(True, linestyle='--', alpha=0.6); plt.legend(); plt.tight_layout()
plt.show()

C:\Users\tommo\AppData\Local\Temp\ipykernel_154436\2299320739.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# In-/Out-Grad-Verteilungen
in_degrees = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.hist(in_degrees, bins=range(max(in_degrees) + 2), color='skyblue', edgecolor='black', align='left')
plt.title('In-Degree Distribution', size=14); plt.xlabel('In-Degree'); plt.ylabel('Number of Nodes')
plt.grid(axis='y', linestyle='--', alpha=0.7); plt.yscale('log')
plt.subplot(1, 2, 2)
plt.hist(out_degrees, bins=range(max(out_degrees) + 2), color='lightcoral', edgecolor='black', align='left')
plt.title('Out-Degree Distribution', size=14); plt.xlabel('Out-Degree'); plt.ylabel('Number of Nodes')
plt.grid(axis='y', linestyle='--', alpha=0.7); plt.yscale('log')
plt.tight_layout()
plt.show()

C:\Users\tommo\AppData\Local\Temp\ipykernel_154436\187482665.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Kapazität vs. Free Flow Time
plt.figure(figsize=(10, 8))
plt.scatter(edges_df['Capacity'], edges_df['free_flow_time'], alpha=0.5, s=10)
plt.title('Capacity vs. Free Flow Time', size=15)
plt.xlabel('Capacity', size=12); plt.ylabel('Free Flow Time', size=12)
if edges_df['Capacity'].min() > 0:
    plt.xscale('log')
if edges_df['free_flow_time'].min() > 0:
    plt.yscale('log')
plt.grid(True, linestyle='--', alpha=0.6); plt.tight_layout()
plt.show()

C:\Users\tommo\AppData\Local\Temp\ipykernel_154436\1581469461.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Verteilung der Link-Typen
link_type_counts = edges_df['Type'].value_counts().sort_index()
plt.figure(figsize=(10, 6))
link_type_counts.plot(kind='bar', color='purple', edgecolor='black', alpha=0.7)
plt.title('Distribution of Edge Link Types', size=15)
plt.xlabel('Link Type', size=12); plt.ylabel('Number of Edges', size=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7); plt.tight_layout()
plt.show()

C:\Users\tommo\AppData\Local\Temp\ipykernel_154436\71154345.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Feature-Konfiguration

Hier wird festgelegt, was in `data.x` (Knoten) und `data.edge_attr` (Kanten) landet.
**Das ist die zentrale Stellschraube für Experimente an der Datenmanipulation.**

Verfügbare Knotenfeatures (Registry in `gnn_pipeline/features.py`):

| Name | Spalten | Beschreibung |
|---|---|---|
| `coords` | `x`, `y` | Koordinaten (Baseline aus dem ursprünglichen GATv2-Notebook) |
| `node_type` | `is_zone` | 1 = Zone (Knoten 1..`<NUMBER OF ZONES>` laut TNTP-Kopf), 0 = Kreuzung |
| `degree` | `log_in_degree`, `log_out_degree` | Grad auf dem vollen Graphen (siehe Leakage-Hinweis im Docstring) |
| `capacity_stats` | `log_capacity_in`, `log_capacity_out` | Summe realer Kapazität ein-/ausgehend |

Alle gewählten Knotenfeatures werden mit `StandardScaler` skaliert (`scale_node_features=True`).

**Kantenfeatures:** Die Platzhalter-Kapazität 999999 der Zonen-Konnektoren wird durch 0 ersetzt (die
Information steckt in `capacity_is_placeholder`), `capacity_ratio` wird nur aus echten Kapazitäten berechnet.
Die numerischen Spalten werden mit `log1p` gestaucht (`log_edge_numerical=True`) und dann standardisiert
(`scale_edge_numerical=True`). Vorher hat der Platzhalter die Skalierung dominiert, und alle echten
Kapazitäten lagen nach dem StandardScaler praktisch auf demselben Wert.

In [8]:
print("Registrierte Knotenfeatures:", sorted(NODE_FEATURE_BUILDERS))

config = FeatureConfig(
    # Baseline: nur Koordinaten. Für Experimente z. B.:
    # node_features=["coords", "node_type", "degree"],
    node_features=["coords"],
    scale_node_features=True,

    edge_numerical=["Capacity", "Length", "free_flow_time", "Toll", "capacity_ratio", "delta_x", "delta_y"],
    edge_binary=["capacity_is_placeholder", "has_reciprocal"],
    one_hot_link_type=True,
    log_edge_numerical=True,     # log1p gegen die starke Rechtsschiefe (Kapazität, Länge, ...)
    scale_edge_numerical=True,   # StandardScaler nach dem log1p
)

SEED = 42
VAL_RATIO, TEST_RATIO, NEG_RATIO = 0.1, 0.1, 2.0
SUPERVISION_RATIO = 0.3   # Anteil der Train-Paare, die nur als Trainingslabels dienen (nicht im Graphen)
NEG_STRATEGY = "hole"     # Negative mit "Loch" (siehe Split-Erklärung und Baselines unten)
SPATIAL_K = 20            # Suchradius für Negative: die 20 räumlich nächsten Knoten

Registrierte Knotenfeatures: ['capacity_stats', 'coords', 'degree', 'node_type']


## Eigenes Knotenfeature ergänzen

Ein neues Feature braucht nur eine Funktion `(nodes_df, edges_df) -> DataFrame` mit einer
Zeile pro Knoten (Index wie `nodes_df`). Entweder dauerhaft in `gnn_pipeline/features.py`
oder zum Ausprobieren direkt hier im Notebook:

In [9]:
from gnn_pipeline import register_node_feature
import numpy as np

@register_node_feature("toll_share")
def _toll_share(nodes_df, edges_df):
    """Anteil mautpflichtiger ausgehender Kanten pro Knoten."""
    has_toll = edges_df.assign(toll=(edges_df["Toll"] > 0).astype(float))
    share = has_toll.groupby("from_node")["toll"].mean()
    return pd.DataFrame({"toll_share": nodes_df["node_id"].map(share).fillna(0).values}, index=nodes_df.index)

# Aktivieren durch: config.node_features.append("toll_share")
print("Registrierte Knotenfeatures:", sorted(NODE_FEATURE_BUILDERS))

Registrierte Knotenfeatures: ['capacity_stats', 'coords', 'degree', 'node_type', 'toll_share']


# Graph bauen, splitten, speichern

In [10]:
data, edges_feat_df, nodes_sorted, node_cols, edge_cols = build_graph(frames, config)

print(data)
print("Knotenfeatures:", node_cols)
print("Kantenfeatures:", edge_cols)
display(edges_feat_df.head())

Data(x=[13389, 2], edge_index=[2, 40003], edge_attr=[40003, 17], pos=[13389, 2], num_nodes=13389)
Knotenfeatures: ['x', 'y']
Kantenfeatures: ['Capacity', 'Length', 'free_flow_time', 'Toll', 'capacity_ratio', 'delta_x', 'delta_y', 'capacity_is_placeholder', 'has_reciprocal', 'link_type_1', 'link_type_2', 'link_type_3', 'link_type_4', 'link_type_6', 'link_type_7', 'link_type_8', 'link_type_9']


,from_node,to_node,Capacity,Length,free_flow_time,Toll,capacity_is_placeholder,link_type_1,link_type_2,link_type_3,...,link_type_6,link_type_7,link_type_8,link_type_9,has_reciprocal,capacity_ratio,delta_x,delta_y,from_node_mapped,to_node_mapped
0,1,4067,0.0,0.12,0.0,0.0,1,False,False,False,...,False,True,False,False,1,0.0,3.0,2.0,0,4066
1,1,4073,0.0,0.12,0.0,0.0,1,False,False,False,...,False,True,False,False,1,0.0,1.0,10.0,0,4072
2,1,4075,0.0,0.12,0.0,0.0,1,False,False,False,...,False,True,False,False,1,0.0,3.0,4.0,0,4074
3,1,8400,0.0,0.12,0.0,0.0,1,False,False,False,...,False,True,False,False,1,0.0,4.0,3.0,0,8399
4,2,4059,0.0,0.12,0.0,0.0,1,False,False,False,...,False,True,False,False,1,0.0,1.0,3.0,1,4058


## Split: paarweise, ein gemeinsamer Graph, Negative mit „Loch“

**1. Paarweise statt kantenweise (reziproker Leak).** Der ursprüngliche Split (`RandomLinkSplit(is_undirected=False)`)
hat jede *gerichtete* Kante unabhängig gezogen. Das Philadelphia-Netz ist aber fast vollständig reziprok
(~94 % der Kanten haben eine Gegenkante). Für **76.8 % der Val- und 84.6 % der Test-Kanten** A→B lag die
Gegenrichtung B→A im Message-Passing-Graphen (Kontrollzelle unten). Jetzt werden **ungeordnete Knotenpaare**
{u, v} aufgeteilt, beide Richtungen einer Straße landen im selben Split.

**2. Ein gemeinsamer Message-Passing-Graph.** Die Train-Paare werden nochmals geteilt: 70 % bilden den Graphen,
30 % (`SUPERVISION_RATIO`) dienen nur als Trainingslabels (wie `disjoint_train_ratio` bei `RandomLinkSplit`).
Train, Val und Test rechnen auf *demselben* Graphen, und keine zu bewertende Kante liegt selbst darin.
- Vorher waren die Trainingslabels zugleich Graphkanten: Das Modell konnte „ist dst schon mein Nachbar?“
  lernen, eine Abkürzung, die bei Val/Test nicht funktioniert.
- Unterschiedlich dichte Graphen für Train/Val/Test hatten die Reihenfolge Train < Val < Test erzeugt und die
  BatchNorm-Statistiken zwischen Training und Auswertung verschoben (springende Loss-Kurven).

**3. Negative mit „Loch“ (`NEG_STRATEGY = "hole"`).** Jede entfernte Kante hinterlässt an ihren Endpunkten
einen geringeren Grad. Bei beliebig gezogenen Negativen reicht deshalb schon „niedriger Grad = Link“ für eine
AUC von ≈ 0.71, und die reine Luftlinie trennt zufällige Paare sogar mit 0.998 (Baselines unten). Ein Modell
lernt dann Löcher und Entfernungen statt Straßenstruktur. Bei `"hole"` stammen beide Endpunkte eines Negativs
aus den Endpunkten der Positiven desselben Splits und liegen unter den 20 räumlich nächsten Knoten. Positive
und Negative haben damit gleich viele Löcher und ähnliche Abstände.

Positive Label = die tatsächlich existierenden gerichteten Kanten (Einbahnstraßen nur in ihrer Richtung);
Negative werden in beide Richtungen eingetragen, neg/pos = `NEG_RATIO`. Der alte kantenweise Split bleibt
über `pair_aware=False` reproduzierbar (Kontrollzelle unten).

In [11]:
PAIR_AWARE = True   # False = alter kantenweiser RandomLinkSplit (reziproker Leak, nur zum Vergleich)

train_data, val_data, test_data = split_graph(
    data, seed=SEED, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO,
    neg_sampling_ratio=NEG_RATIO, pair_aware=PAIR_AWARE, neg_strategy=NEG_STRATEGY,
    spatial_k=SPATIAL_K, supervision_ratio=SUPERVISION_RATIO,
)

bundle = GraphBundle(
    data=data, train=train_data, val=val_data, test=test_data,
    node_feature_columns=node_cols, edge_attr_columns=edge_cols,
    node_type=nodes_sorted["type"].tolist(),
    original_node_ids=nodes_sorted["node_id"].tolist(),
    meta={"seed": SEED, "val_ratio": VAL_RATIO, "test_ratio": TEST_RATIO,
          "neg_sampling_ratio": NEG_RATIO, "neg_strategy": NEG_STRATEGY, "spatial_k": SPATIAL_K,
          "supervision_ratio": SUPERVISION_RATIO,
          "node_features": list(config.node_features),
          "split": "pair_aware" if PAIR_AWARE else "directed_random_link_split",
          "log_edge_numerical": config.log_edge_numerical,
          "scale_edge_numerical": config.scale_edge_numerical},
)
print(bundle.summary())
save_bundle(bundle)   # -> processed/philadelphia_split.pt

Knoten: 13389 (1525 Zonen), Kanten: 40003
Knotenfeatures (2): ['x', 'y']
Kantenfeatures (17): ['Capacity', 'Length', 'free_flow_time', 'Toll', 'capacity_ratio', 'delta_x', 'delta_y', 'capacity_is_placeholder', 'has_reciprocal', 'link_type_1', 'link_type_2', 'link_type_3', 'link_type_4', 'link_type_6', 'link_type_7', 'link_type_8', 'link_type_9']
train: msg-passing Kanten=22416, Label-Paare pos=9579 neg=19158
val: msg-passing Kanten=22416, Label-Paare pos=3999 neg=7998
test: msg-passing Kanten=22416, Label-Paare pos=4009 neg=8018
train: positive Label-Kanten im Message-Passing-Graph: 0/9579, davon Gegenrichtung im Graph: 0/9579 (0.0%)
val: positive Label-Kanten im Message-Passing-Graph: 0/3999, davon Gegenrichtung im Graph: 0/3999 (0.0%)
test: positive Label-Kanten im Message-Passing-Graph: 0/4009, davon Gegenrichtung im Graph: 0/4009 (0.0%)
meta: {'seed': 42, 'val_ratio': 0.1, 'test_ratio': 0.1, 'neg_sampling_ratio': 2.0, 'neg_strategy': 'hole', 'spatial_k': 20, 'supervision_ratio': 0.

'processed\\philadelphia_split.pt'

In [12]:
# Kontrolle: keine bewertete Kante (und keine Gegenrichtung) liegt im Message-Passing-Graph;
# zum Vergleich der reziproke Leak im alten kantenweisen Split
from gnn_pipeline import reverse_edge_leakage
from gnn_pipeline.graph import edges_in_graph

for name, split in [("train", train_data), ("val", val_data), ("test", test_data)]:
    assert edges_in_graph(split, split) == 0, f"{name}: Label-Kanten liegen im Message-Passing-Graph"
    leaked, total = reverse_edge_leakage(split, split)
    assert leaked == 0, f"{name}: {leaked} positive Kanten haben ihre Gegenrichtung im Message-Passing-Graph"
print("Neuer Split: keine Label-Kante und keine Gegenrichtung im gemeinsamen Message-Passing-Graph.")

old_train, old_val, old_test = split_graph(
    data, seed=SEED, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO, neg_sampling_ratio=NEG_RATIO, pair_aware=False
)
for name, split in [("val", old_val), ("test", old_test)]:
    leaked, total = reverse_edge_leakage(split, split)
    print(f"Alter Split ({name}): {leaked}/{total} = {leaked / total:.1%} der positiven Kanten geleakt")

Neuer Split: keine Label-Kante und keine Gegenrichtung im gemeinsamen Message-Passing-Graph.


Alter Split (val): 3074/4000 = 76.8% der positiven Kanten geleakt
Alter Split (test): 3383/4000 = 84.6% der positiven Kanten geleakt


## Heuristik-Baselines: Wie schwer ist die Aufgabe ohne Lernen?

ROC-AUC klassischer Link-Prediction-Heuristiken auf denselben Label-Paaren (berechnet auf dem gemeinsamen
Message-Passing-Graphen). Ein GNN muss sich an diesen Werten messen lassen. Verglichen werden die drei
Negativ-Strategien:

- **-Distanz (Luftlinie):** Bei `"uniform"` ≈ 0.998 – zufällige Knotenpaare liegen fast immer weit
  auseinander, die Aufgabe ist trivial. Bei `"hole"` ≈ 0.72, in allen drei Splits gleich.
- **Niedriger Grad (Loch):** Bei `"uniform"`/`"spatial"` ≈ 0.71 – die entfernte Kante verrät sich durch den
  geringeren Grad ihrer Endpunkte. Bei `"hole"` ≈ 0.53, das Signal ist neutralisiert.
- **Gemeinsame Nachbarn / Adamic-Adar:** Straßennetze haben kaum Dreiecke, daher ≤ 0.5.

`"hole"` ist damit die einzige Variante, bei der weder Entfernung noch Grad allein die Aufgabe lösen.

In [13]:
from gnn_pipeline import heuristic_baselines

for strategy in ("uniform", "spatial", "hole"):
    if strategy == NEG_STRATEGY:
        splits = (train_data, val_data, test_data)
    else:
        splits = split_graph(data, seed=SEED, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO,
                             neg_sampling_ratio=NEG_RATIO, neg_strategy=strategy,
                             spatial_k=SPATIAL_K, supervision_ratio=SUPERVISION_RATIO)
    print(f"neg_strategy={strategy!r}" + ("  <- verwendet" if strategy == NEG_STRATEGY else ""))
    display(heuristic_baselines(*splits).round(3))

neg_strategy='uniform'

,train,val,test
Heuristik,,,
-Distanz (Koordinaten),0.998,0.998,0.998
Gemeinsame Nachbarn,0.507,0.505,0.503
Adamic-Adar,0.507,0.505,0.503
Niedriger Grad (Loch),0.710,0.705,0.696


neg_strategy='spatial'


,train,val,test
Heuristik,,,
-Distanz (Koordinaten),0.714,0.725,0.727
Gemeinsame Nachbarn,0.451,0.449,0.454
Adamic-Adar,0.451,0.449,0.454
Niedriger Grad (Loch),0.720,0.714,0.716


neg_strategy='hole'  <- verwendet


,train,val,test
Heuristik,,,
-Distanz (Koordinaten),0.708,0.724,0.724
Gemeinsame Nachbarn,0.474,0.477,0.474
Adamic-Adar,0.474,0.477,0.474
Niedriger Grad (Loch),0.539,0.528,0.541
